In [38]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv("urinalysis_tests.csv")

print(df.shape)

df.head()

(1436, 16)


,Unnamed: 0,Age,Gender,Color,Transparency,Glucose,Protein,pH,Specific Gravity,WBC,RBC,Epithelial Cells,Mucous Threads,Amorphous Urates,Bacteria,Diagnosis
0,0,76.0,FEMALE,LIGHT YELLOW,CLEAR,NEGATIVE,NEGATIVE,5.0,1.010,1-3,0-2,OCCASIONAL,RARE,NONE SEEN,OCCASIONAL,NEGATIVE
1,1,9.0,MALE,DARK YELLOW,SLIGHTLY HAZY,NEGATIVE,1+,5.0,1.030,1-3,0-2,RARE,FEW,FEW,MODERATE,NEGATIVE
2,2,12.0,MALE,LIGHT YELLOW,SLIGHTLY HAZY,NEGATIVE,TRACE,5.0,1.030,0-3,0-2,RARE,FEW,MODERATE,RARE,NEGATIVE
3,3,77.0,MALE,BROWN,CLOUDY,NEGATIVE,1+,6.0,1.020,5-8,LOADED,RARE,RARE,NONE SEEN,FEW,NEGATIVE
4,4,29.0,FEMALE,YELLOW,HAZY,NEGATIVE,TRACE,6.0,1.025,1-4,0-2,RARE,RARE,NONE SEEN,FEW,NEGATIVE


In [3]:
print(df.isnull().sum())

Unnamed: 0          0
Age                 0
Gender              0
Color               1
Transparency        0
Glucose             0
Protein             0
pH                  0
Specific Gravity    0
WBC                 0
RBC                 0
Epithelial Cells    0
Mucous Threads      0
Amorphous Urates    0
Bacteria            0
Diagnosis           0
dtype: int64


In [4]:
df = df.fillna("UNKNOWN")
print(df.isnull().sum())

Unnamed: 0          0
Age                 0
Gender              0
Color               0
Transparency        0
Glucose             0
Protein             0
pH                  0
Specific Gravity    0
WBC                 0
RBC                 0
Epithelial Cells    0
Mucous Threads      0
Amorphous Urates    0
Bacteria            0
Diagnosis           0
dtype: int64


In [5]:
print(df["Diagnosis"].value_counts())

Diagnosis
NEGATIVE    1355
POSITIVE      81
Name: count, dtype: int64


In [6]:
print(df["Protein"].value_counts())
protein_map = {
    "NEGATIVE": 0,
    "TRACE": 0.5,
    "1+": 1,
    "2+": 2,
    "3+": 3,
    "4+": 4
}

df["Protein"] = df["Protein"].map(protein_map)

Protein
NEGATIVE    804
TRACE       492
1+           94
2+           41
3+            5
Name: count, dtype: int64


In [7]:
print(df["Glucose"].value_counts())
glucose_map = {
    "NEGATIVE": 0,
    "TRACE": 0.5,
    "1+": 1,
    "2+": 2,
    "3+": 3,
    "4+": 4
}

df["Glucose"] = df["Glucose"].map(glucose_map)

Glucose
NEGATIVE    1349
2+            24
3+            23
1+            15
TRACE         13
4+            12
Name: count, dtype: int64


In [8]:
print(df["Bacteria"].value_counts())
bacteria_map = {
    "NONE SEEN":0,
    "RARE":1,
    "FEW":2,
    "OCCASIONAL":3,
    "MODERATE":4,
    "PLENTY":5,
    "LOADED":6
}

df["Bacteria"] = df["Bacteria"].map(bacteria_map)

Bacteria
RARE          755
FEW           434
MODERATE      158
PLENTY         77
OCCASIONAL      8
LOADED          4
Name: count, dtype: int64


In [9]:
print(df["Epithelial Cells"].value_counts())
cell_map = {
    "NONE SEEN":0,
    "RARE":1,
    "FEW":2,
    "OCCASIONAL":3,
    "MODERATE":4,
    "PLENTY":5,
    "LOADED":6
}

df["Epithelial Cells"] = df["Epithelial Cells"].map(cell_map)

Epithelial Cells
RARE          742
FEW           347
MODERATE      188
PLENTY        121
OCCASIONAL     19
NONE SEEN      16
LOADED          3
Name: count, dtype: int64


In [10]:
def convert_range(value):
    value = str(value).strip()
    if ">" in value:
        return float(value.replace(">", ""))
    if "-" in value:
        low, high = value.split("-")
        return (float(low) + float(high)) / 2
    try:
        return float(value)
    except:
        return None

In [11]:
df["WBC"] = df["WBC"].apply(convert_range)
df["RBC"] = df["RBC"].apply(convert_range)

In [12]:
from sklearn.preprocessing import LabelEncoder
categorical_cols =['Gender','Color','Transparency','Amorphous Urates','Mucous Threads']
encoders={}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

In [13]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1436 non-null   int64  
 1   Age               1436 non-null   float64
 2   Gender            1436 non-null   int64  
 3   Color             1436 non-null   int64  
 4   Transparency      1436 non-null   int64  
 5   Glucose           1436 non-null   float64
 6   Protein           1436 non-null   float64
 7   pH                1436 non-null   float64
 8   Specific Gravity  1436 non-null   float64
 9   WBC               1410 non-null   float64
 10  RBC               1431 non-null   float64
 11  Epithelial Cells  1436 non-null   int64  
 12  Mucous Threads    1436 non-null   int64  
 13  Amorphous Urates  1436 non-null   int64  
 14  Bacteria          1436 non-null   int64  
 15  Diagnosis         1436 non-null   object 
dtypes: float64(7), int64(8), object(1)
memory 

In [15]:
df['Diagnosis'].value_counts()

Diagnosis
NEGATIVE    1355
POSITIVE      81
Name: count, dtype: int64

In [16]:
df['Diagnosis']= df['Diagnosis'].map({'POSITIVE': 1, 'NEGATIVE': 0})

In [17]:
normal_df = df[df["Diagnosis"] == 0].copy()
print(normal_df.shape)

(1355, 16)


Create Features

In [18]:
X_normal = normal_df.drop("Diagnosis",axis=1)

### Standardize Features

In [19]:
scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)

In [20]:
joblib.dump(scaler,"urinalysis_anomaly_scaler.pkl")

['urinalysis_anomaly_scaler.pkl']

### Train Isolation Forest

In [23]:
X_train_normal, X_test_normal = train_test_split(
    X_normal,
    test_size=0.2,
    random_state=42
)

print(X_train_normal.shape)
print(X_test_normal.shape)

(1084, 15)
(271, 15)


In [24]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_normal
)

X_test_scaled = scaler.transform(
    X_test_normal
)

In [26]:
X_all = df.drop("Diagnosis",axis=1)
y_all = df["Diagnosis"]
X_all_scaled = scaler.transform(X_all)

## Hyperparameter Search

In [27]:
param_grid = {

    "contamination": [
        0.01,
        0.03,
        0.05,
        0.07,
        0.10
    ],

    "n_estimators": [
        100,
        200,
        300,
        500
    ],

    "max_samples": [
        "auto",
        0.7,
        0.8,
        1.0
    ]
}

In [28]:
results = []

In [29]:
from sklearn.metrics import f1_score, precision_score, recall_score


for contamination in param_grid["contamination"]:
    for n_estimators in param_grid["n_estimators"]:
        for max_samples in param_grid["max_samples"]:
            iso = IsolationForest(
                contamination=contamination,
                n_estimators=n_estimators,
                max_samples=max_samples,
                random_state=42
            )

            iso.fit(X_train_scaled)
            preds = iso.predict(X_all_scaled)

            preds = np.where(preds == -1,1,0)

            f1 = f1_score(
                y_all,
                preds
            )

            precision = precision_score(
                y_all,
                preds
            )

            recall = recall_score(y_all,preds)

            results.append({
                "contamination":contamination,
                "n_estimators":n_estimators,
                "max_samples":max_samples,
                "f1":f1,
                "precision":precision,
                "recall":recall
            })

In [30]:
results_df = pd.DataFrame(
    results
)

results_df.sort_values(
    by="f1",
    ascending=False
).head(20)

,contamination,n_estimators,max_samples,f1,precision,recall
23,0.03,200,1.0,0.289855,0.350877,0.246914
27,0.03,300,1.0,0.289855,0.350877,0.246914
28,0.03,500,auto,0.289855,0.350877,0.246914
31,0.03,500,1.0,0.289855,0.350877,0.246914
16,0.03,100,auto,0.287770,0.344828,0.246914
50,0.07,100,0.8,0.285714,0.243478,0.345679
24,0.03,300,auto,0.285714,0.338983,0.246914
29,0.03,500,0.7,0.279412,0.345455,0.234568
51,0.07,100,1.0,0.275510,0.234783,0.333333
26,0.03,300,0.8,0.275362,0.333333,0.234568


In [31]:
best_row = results_df.sort_values(
    by="f1",
    ascending=False
).iloc[0]

print(best_row)

contamination        0.03
n_estimators          200
max_samples           1.0
f1               0.289855
precision        0.350877
recall           0.246914
Name: 23, dtype: object


In [32]:
best_iso = IsolationForest(

    contamination=
    best_row["contamination"],
    n_estimators=
    int(best_row["n_estimators"]),
    max_samples=
    best_row["max_samples"],
    random_state=42
)

best_iso.fit(X_train_scaled)

IsolationForest(contamination=np.float64(0.03), max_samples=1.0,
                n_estimators=200, random_state=42)

## Generate Anomaly Prediction

In [33]:
anomaly_preds = best_iso.predict(X_all_scaled)

In [34]:
anomaly_preds = np.where(anomaly_preds == -1,1,0)

In [35]:
print(anomaly_preds)

[0 1 0 ... 0 0 0]


#### Evaluation metrics

In [36]:
print(confusion_matrix(y_all,anomaly_preds))

[[1318   37]
 [  61   20]]


In [39]:
print(classification_report(y_all,anomaly_preds))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96      1355
           1       0.35      0.25      0.29        81

    accuracy                           0.93      1436
   macro avg       0.65      0.61      0.63      1436
weighted avg       0.92      0.93      0.93      1436



### Anomaly Scores

In [40]:
scores = best_iso.decision_function(X_all_scaled)

In [41]:
print(scores)

[ 0.06039932 -0.00442107  0.03037516 ...  0.10297096  0.04397034
  0.14569569]


In [42]:
anomaly_score = -scores

In [43]:
df["AnomalyScore"] = anomaly_score

### Inspect Top Anomalies

In [44]:
df.sort_values(by="AnomalyScore",ascending=False)[["Diagnosis","AnomalyScore"]].head(20)

,Diagnosis,AnomalyScore
611,0,0.124202
1397,0,0.118652
505,0,0.080716
934,0,0.063091
422,1,0.061558
581,0,0.061493
244,1,0.058913
805,0,0.057095
960,0,0.055601
592,1,0.055484


In [45]:
joblib.dump(
    best_iso,
    "urinalysis_isolation_forest.pkl"
)

['urinalysis_isolation_forest.pkl']

In [46]:
joblib.dump(
    scaler,
    "urinalysis_isolation_scaler.pkl"
)

['urinalysis_isolation_scaler.pkl']